### stacking

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import StackingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report
import seaborn as sns
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder

In [6]:
df = sns.load_dataset('iris')

In [7]:
df.head()

,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


In [8]:
X = df.drop('species', axis =1)
y = df['species']

In [9]:
le = LabelEncoder()
y_encoded = le.fit_transform(y)

In [10]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42, stratify = y_encoded)

In [11]:
base_learners = [
    ('dt', DecisionTreeClassifier(random_state=42)),
    ('svc',SVC(probability=True, kernel='rbf',random_state=42)),
    ('lr',LogisticRegression(max_iter=1000))
]

In [12]:
meta_learner = LogisticRegression(max_iter=1000)

In [13]:
stacking_clf = StackingClassifier(
    estimators=base_learners,
    final_estimator=meta_learner,
    cv=5
)

In [14]:
stacking_clf.fit(X_train,y_train)

StackingClassifier(cv=5,
                   estimators=[('dt', DecisionTreeClassifier(random_state=42)),
                               ('svc', SVC(probability=True, random_state=42)),
                               ('lr', LogisticRegression(max_iter=1000))],
                   final_estimator=LogisticRegression(max_iter=1000))

In [15]:
y_pred = stacking_clf.predict(X_test)
y_pred

array(['setosa', 'virginica', 'versicolor', 'versicolor', 'setosa',
       'versicolor', 'setosa', 'setosa', 'virginica', 'versicolor',
       'virginica', 'virginica', 'virginica', 'versicolor', 'setosa',
       'setosa', 'setosa', 'versicolor', 'versicolor', 'virginica',
       'setosa', 'virginica', 'versicolor', 'virginica', 'virginica',
       'virginica', 'versicolor', 'setosa', 'virginica', 'setosa'],
      dtype=object)

In [16]:
y_test

,species
38,setosa
127,virginica
57,versicolor
93,versicolor
42,setosa
56,versicolor
22,setosa
20,setosa
147,virginica
84,versicolor


In [17]:
accuracy = accuracy_score(y_test,y_pred)
accuracy

0.9666666666666667

### Bagging


### random forest

In [18]:
from sklearn.ensemble import RandomForestClassifier

In [19]:
rf_model = RandomForestClassifier(
    n_estimators = 100, ##number of tress
    max_depth = None,
    random_state = 42
)

In [20]:
rf_model.fit(X_train,y_train)

RandomForestClassifier(random_state=42)

In [21]:
y_pred = rf_model.predict(X_test)
y_pred

array(['setosa', 'virginica', 'versicolor', 'versicolor', 'setosa',
       'versicolor', 'setosa', 'setosa', 'virginica', 'versicolor',
       'virginica', 'virginica', 'virginica', 'versicolor', 'setosa',
       'setosa', 'setosa', 'versicolor', 'versicolor', 'versicolor',
       'setosa', 'virginica', 'versicolor', 'versicolor', 'virginica',
       'virginica', 'versicolor', 'setosa', 'virginica', 'setosa'],
      dtype=object)

In [22]:
accuracy_score(y_test,y_pred)

0.9

### boosting

In [23]:
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier

In [24]:
from xgboost import XGBClassifier

In [25]:
ada_model = AdaBoostClassifier(n_estimators=100, random_state=42)

In [26]:
ada_model.fit(X_train,y_train)

AdaBoostClassifier(n_estimators=100, random_state=42)

In [27]:
y_pred = ada_model.predict(X_test)

In [28]:
accuracy = accuracy_score(y_test,y_pred)

In [29]:
accuracy

0.9333333333333333

### gradient boosting model

In [30]:
gb_model = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1,random_state=42)

In [31]:
gb_model.fit(X_train,y_train)

GradientBoostingClassifier(random_state=42)

In [32]:
y_pred = gb_model.predict(X_test)

In [34]:
accuracy = accuracy_score(y_test,y_pred)
accuracy

0.9666666666666667

### XGBOOST

In [44]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)

In [45]:
xgb_model = XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=3,use_label_encoder=False, eval_metric='mlogloss',random_state=42)

In [47]:
xgb_model.fit(X_train, y_train_encoded)

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [13:48:22] WARNING: /__w/xgboost/xgboost/src/learner.cc:793: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric='mlogloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=3, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=100, n_jobs=None,
              num_parallel_tree=None, ...)

In [52]:
y_pred_xgb = xgb_model.predict(X_test)

In [53]:
accuracy = accuracy_score(y_test_encoded,y_pred_xgb)
accuracy

0.9333333333333333